In [22]:
import os
import time
from dataclasses import dataclass
from IPython.display import display, Markdown

from langchain_community.vectorstores import Chroma
from langchain_core.tools import create_retriever_tool
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document
from typing import List
from datetime import datetime

In [24]:
vector_store = Chroma(persist_directory="vector_db/ircc_cleaned", embedding_function=OllamaEmbeddings(model="qwen3-embedding:0.6b"))


In [25]:
print("Total documents:", vector_store._collection.count())

Total documents: 9300


In [26]:
@dataclass
class Context:
    user_id: str

llm = ChatOllama(model="qwen2.5:14b-instruct", temperature=0.2) # keep_alive=-1
# llm = ChatGoogleGenerativeAI( model="gemini-2.0-flash", temperature=0, max_tokens=None, timeout=None, max_retries=2) # Or "gemini-pro"

In [27]:
import math

In [41]:
class LatestFirstRetriever(BaseRetriever):
    vector_store: Chroma
    k: int = 5

    def _get_relevant_documents(self, query: str) -> List[Document]:
        docs_with_score = self.vector_store.similarity_search_with_score(
            query,
            k=20,
            filter={"archived": False}
        )

        ranked = []

        for doc, score in docs_with_score:

            similarity_score = 1 / (1 + score)

            date_obj = datetime.strptime(doc.metadata["date"], "%Y-%m-%d")
            days_old = (datetime.now() - date_obj).days
            recency_score = math.exp(-days_old / 365)

            
            # recency_score = date_object.timestamp()
            final_score = (similarity_score * 0.8) + (recency_score * 0.2)
            doc.page_content = f"""date: {doc.metadata.get("date")}\n URL: {doc.metadata.get("url")}\n Content: {doc.page_content}\n\n."""
            ranked.append((doc, final_score))

            # ranked.append((doc, final_score, score, similarity_score * 0.8, recency_score * 0.2))

        ranked.sort(key=lambda x: x[1], reverse=True)        
        return [d for d, _ in ranked[:self.k]]

        

retriever = LatestFirstRetriever(vector_store=vector_store, k=20)

retriever_tool = create_retriever_tool(
    retriever,
    name="immigration_knowladge_base_search",
    description="This database has all the knowledge regarding Canada immigration programs."
)

ValidationError: 1 validation error for LatestFirstRetriever
vector_store
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing

In [ ]:
hybrid_retriever.search(query, k=5)

In [30]:
# query = "what are acceptable english language test results in express entry?"

# docs = retriever.invoke(query)

# print(f"Returned {len(docs)} documents\n")

# # for i, (d, ns, os, ss, rs) in enumerate(docs):
# #     print(f"--- Document {i+1}: New score:{ns:.4f}: Old Score {os:.4f}:  ss {ss:.4f}:  rs {rs:.4f} ---")
# for i, (d) in enumerate(docs):
#     print(f"--- Document {i+1}---")
#     # print("Date:", d.metadata.get("date"))
#     # print("Archived:", d.metadata.get("archived"))
#     # print("URL:", d.metadata.get("url"))
#     # print("Program:", d.metadata.get("category"))
#     # print(d.page_content[:300])
#     print(d.page_content)

#     print()

In [36]:
agent = create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt="""You are a professional Canada Immigration consultant AI. 

Rules:
1. You must retrieve information from immigration_knowladge_base_search tool before answering the question and Only use retrieved information.
2. Do NOT create, guess, or hallucinate URLs, program names, or regulations.
3. If the answer is not in the retrieved documents, respond: "I don't have sufficient information to answer this question."
4. Always cite the URLs exactly as they appear in the documents.
5. Answer the response in same language as of question.

Always:
- Provide clear explanation
- Mention program name (only if present in retrieved documents)
- Add citation URL from the retrieved documents
        """,
    context_schema=Context,
    checkpointer=InMemorySaver()
)

In [37]:
def summarize_history(p_s, old_msgs):
    print("summarizing old conversations")
    prompt = f"Summarize the following conversation concisely:\n\nPrevious summary: {p_s}\n\nNew coversation: "
    for msg in old_msgs:
        prompt += f"{msg['role'].capitalize()}: {msg['content']}\n"
    response = llm.invoke(prompt)
    return f"Summary of previous conversation: {response.content}"

In [33]:
from IPython.display import display, Markdown

def stream_agent(agent, messages, user_id):
    for chunk in agent.stream(
        {"messages": messages},
        config={'configurable': {'thread_id': 1}},
        # context=Context(user_id=user_id),
        stream_mode="values"
    ):
        # Only handle assistant text
        if chunk.get("role") == "assistant" and "content" in chunk:
            display(Markdown(chunk['content']))

In [34]:

MAX_HISTORY = 3  # last detailed messages
summary = ""     # summary of older conversation

# chat_history = []


In [ ]:

# display(Markdown(chat_history[0]['content']))

In [38]:
user_input = input("Enter: ")
while user_input.lower() != 'exit':
    # if len(chat_history)>MAX_HISTORY:
    #     summary = summarize_history(summary, chat_history)
    #     chat_history = []
    #     chat_history.append({'role':'system', 'content': summary})

    # chat_history.append({'role':'user', 'content':user_input})
    response = agent.invoke(
        # {'messages': chat_history},
        {'messages': {'role':'user', 'content':user_input}},

        config={'configurable':{'thread_id':1}},
        context=Context(user_id='ankit1')
    )
    assistant_msg = response['messages'][-1].content
    display(Markdown(assistant_msg))
    # stream_agent(agent, chat_history, user_id='ankit1')
    user_input = input("Enter: ")

Enter:  Hello my name is Ankit. I speak English. I have one year of Canadian experience what is the best programme for me to get PR?


To provide you with the most accurate advice, could you please specify your occupation or the type of work you did in Canada? This information will help determine which immigration program might be suitable for you.

However, based on your one year of Canadian experience, here are a few programs that may apply:

1. **Federal Skilled Trades Program (FSTP)**: If your job is related to skilled trades.
2. **Canadian Experience Class (CEC)**: This program targets foreign workers who have Canadian work experience and meet the necessary requirements.

For detailed information on these programs, please refer to:
- [Federal Skilled Trades Program](https://www.canada.ca/en/immigration-refugees-citizenship/services/work-canada/experience-class.html)
- [Canadian Experience Class](https://www.canada.ca/en/immigration-refugees-citizenship/services/work-canada/experience-class.html)

Once you provide more details about your occupation, I can give a more precise recommendation.

Enter:  what is CRS score can you calculate my CRS score? ask me neccessory questions to calcualtes the score


Certainly! The Comprehensive Ranking System (CRS) is used by Immigration, Refugees and Citizenship Canada (IRCC) to evaluate candidates applying under the Express Entry system. To calculate your CRS score, I need specific information about you, including:

1. **Core Human Capital Factors**:
   - Age
   - Education level and Canadian educational equivalency
   - Language proficiency in English or French (CLB/NCLC scores)
   - Work experience

2. **Additional Points for Skilled Trades and Provincial Nominee Programs (PNP)**:
   - Whether you have a job offer from a Canadian employer
   - If you are nominated by a province under the PNP

3. **Spouse Information** (if applicable):
   - Spouse's age, education level, language proficiency, and work experience

Please provide this information so I can calculate your CRS score.

For more details on how each factor is scored, refer to:
- [Comprehensive Ranking System](https://www.canada.ca/en/immigration-refugees-citizenship/services/application/express-entry-system/ee-points.html)

Let's start with your age and education level. Please provide these details first.

Enter:  here is my details calculate my score. Age – 31. Language Test Results – IELTS . Scores for English (speaking, reading , writing, listening) are 7, 7, 7, 8. Education Details – Master’s. The duration of study 2 year. Work Experience – Canada 1 year, abroad 1 year. NOC (National Occupational Classification) level of  job: 1 – Professional. Years of work experience in Canada (if applicable). 1 year in canada. Spouse/Partner Details (if applicable) – unmarried . If your spouse is a Canadian citizen or permanent resident, you may get additional points.- No.  Job Offer in Canada (if applicable) – No. If you have a job offer from a designated employer, you may get extra points. - No. calculate my CRS Score for express entry: canadian experiance class programme.


...

Enter:  is there any provincial progamme where i can be eligible ?


Based on your details, you may also be eligible for some Provincial Nominee Programs (PNPs). Here are a few PNPs that could potentially fit your profile:

1. **Alberta Express Entry Stream**: This stream targets skilled workers who have an Alberta job offer or work experience in Alberta.
2. **British Columbia PNP Tech Pilot Program**: If you work in the tech sector, this program may be suitable.
3. **Ontario Immigrant Nominee Program (OINP) – Human Capital Stream**: This stream is for skilled workers with a valid Express Entry profile and who meet Ontario’s criteria.

Let's review your eligibility for these programs:

### Alberta Express Entry Stream
- **Job Offer in Alberta**: If you have an Alberta job offer, this would be highly beneficial.
- **Work Experience in Alberta**: One year of work experience in Alberta at the Professional level (NOC 0) could make you eligible.

### British Columbia PNP Tech Pilot Program
- **Tech Sector Work Experience**: You need to have relevant tech sector work experience and a BC job offer.
- **Express Entry Profile**: You must be registered with Express Entry.

### Ontario Immigrant Nominee Program (OINP) – Human Capital Stream
- **Express Entry Profile**: You must be in the federal Express Entry pool.
- **Work Experience**: One year of skilled work experience in Ontario is required.
- **Language Proficiency**: Your IELTS scores are strong, which meets the language requirements.

Given your one year of Canadian work experience and your high IELTS scores, you might be eligible for these programs if you meet additional criteria such as having a job offer or relevant work experience in the province.

For detailed eligibility criteria and application processes, please refer to:
- [Alberta Express Entry Stream](https://www.albertaimmigration.ca/programs/alberta-express-entry-stream/)
- [British Columbia PNP Tech Pilot Program](https://www2.gov.bc.ca/gov/content/employment-business/migration/british-columbia-provincial-nominee-program/pnp-categories/tech-pilot)
- [Ontario Immigrant Nominee Program (OINP) – Human Capital Stream](https://www.ontarioimmigration.ca/nominee-program/human-capital-stream)

If you have a job offer or relevant work experience in any of these provinces, it would be worth applying to their respective PNPs.

Would you like more detailed information on any specific PNP?

Enter:  i heard that there is fifa in canada this year can i apply for my family's visa to watch fifa?


The FIFA World Cup Canada 2026 will be a significant event, and visitors interested in attending the tournament can apply for temporary resident visas (visitor visas) or electronic travel authorizations (eTA), depending on their nationality.

To bring your family to Canada specifically for the FIFA World Cup:

1. **Visitor Visa**: If you are not from a visa-exempt country, you will need to apply for a visitor visa.
2. **Electronic Travel Authorization (eTA)**: Citizens of eligible countries can use eTA instead of a visitor visa.

Here’s what you need to do:

### Visitor Visa Application
- **Application Process**: You or your family members must complete an online application and provide necessary documents such as passport, proof of ties to home country, travel itinerary, financial support, and accommodation details.
- **Processing Time**: Processing times can vary depending on the volume of applications. It’s advisable to apply well in advance.

### Electronic Travel Authorization (eTA)
- **Application Process**: If you are from an eligible country, you can apply for eTA online. The application is straightforward and typically takes a few minutes to complete.
- **Processing Time**: eTA approvals are usually quick, often within minutes or hours.

For more detailed information on the visitor visa process and requirements:

- [Visitor Visa Application](https://www.canada.ca/en/services/immigration-citizenship/application-types.html)
- [Electronic Travel Authorization (eTA)](https://www.canada.ca/en/services/immigration-citizenship/electronic-travel-authorizations.html)

If you need further assistance or have specific questions about the application process, feel free to ask!

Would you like more detailed information on how to apply for a visitor visa or eTA?

KeyboardInterrupt: Interrupted by user

In [ ]:
Hello my name is Ankit. I speak English. I have one year of Canadian experience what is the best programme for me to get PR?
# am I eligible for Express Entry - Canadian experience class? tell me more about that program.
# what is CRS score can you calculate my CRS score? ask me neccessory questions to calcualtes the score
here is my details calculate my score.
Age – 31.
Language Test Results – IELTS .
Scores for English (speaking, reading , writing, listening) are 7, 7, 7, 8.
Education Details – Master’s.
The duration of study 2 year.
Work Experience – Canada 1 year, abroad 1 year.
NOC (National Occupational Classification) level of  job: 1 – Professional.
Years of work experience in Canada (if applicable). 1 year in canada.
Spouse/Partner Details (if applicable) – unmarried .
If your spouse is a Canadian citizen or permanent resident, you may get additional points.- No. 
Job Offer in Canada (if applicable) – No.
If you have a job offer from a designated employer, you may get extra points. - No.
calculate my CRS Score for express entry: canadian experiance class programme.
    
I studied computer science and I am working in AI. is there any programme based on in demand skills?
can i apply for green card in usa?
is there any provincial progamme where i can be eligible ?
i heard that there is fifa in canada this year can i apply for my family's visa to watch fifa?

In [40]:
response['messages']

[HumanMessage(content='Hello my name is Ankit. I speak English. I have one year of Canadian experience what is the best programme for me to get PR?', additional_kwargs={}, response_metadata={}, id='d4ee4745-9f13-43f2-b5e2-af20fa77ee4f'),
 AIMessage(content='To provide you with the most accurate advice, could you please specify your occupation or the type of work you did in Canada? This information will help determine which immigration program might be suitable for you.\n\nHowever, based on your one year of Canadian experience, here are a few programs that may apply:\n\n1. **Federal Skilled Trades Program (FSTP)**: If your job is related to skilled trades.\n2. **Canadian Experience Class (CEC)**: This program targets foreign workers who have Canadian work experience and meet the necessary requirements.\n\nFor detailed information on these programs, please refer to:\n- [Federal Skilled Trades Program](https://www.canada.ca/en/immigration-refugees-citizenship/services/work-canada/experien

In [39]:
from langchain.messages import ToolMessage, AIMessage, HumanMessage, SystemMessage
from pprint import pprint
for m in  response['messages']:
    # pprint(m)
    if isinstance(m, HumanMessage):
        print("".join(['=']*70)+"Human Message"+"".join(['=']*70))
        print(m.content)
    elif isinstance(m, AIMessage):
        print("".join(['=']*70)+"AI Message"+"".join(['=']*70))
        if m.content:
            print(m.content[:200])
        if m.tool_calls:
            print("Tools call:" + m.tool_calls[0]['args']['query'])
    elif isinstance(m, ToolMessage):
        print("".join(['=']*70)+"Tool Message"+"".join(['=']*70))
        print(m.content[:200])
    elif isinstance(m, SystemMessage):
        print("".join(['=']*70)+"System Message"+"".join(['=']*70))
        print(m.content[:200])
    else:
        pprint(m)
    print('\n\n')
# response['messages']

======================================================================Human Message======================================================================
Hello my name is Ankit. I speak English. I have one year of Canadian experience what is the best programme for me to get PR?



======================================================================AI Message======================================================================
To provide you with the most accurate advice, could you please specify your occupation or the type of work you did in Canada? This information will help determine which immigration program might be su



======================================================================Human Message======================================================================
what is CRS score can you calculate my CRS score? ask me neccessory questions to calcualtes the score



======================================================================AI Message========================

In [ ]:
docs[0]

In [ ]:
docs[0]

In [ ]:
query = "how to calculate CRS score for Canadian Experience Class (CEC) in Express Entry"

docs = retriever.invoke(query)

print(f"Returned {len(docs)} documents\n")

for i, (d, ns, os, ss, rs) in enumerate(docs):
    print(f"--- Document {i+1}: New score:{s:.4f}: Old Score {os:.4f}:  ss {ss:.4f}:  rs {rs:.4f} ---")
    print("Date:", d.metadata.get("date"))
    print("Archived:", d.metadata.get("archived"))
    print("URL:", d.metadata.get("url"))
    print("Program:", d.metadata.get("category"))
    # print("Preview:", d.page_content[:300])
    print()